In [ ]:
import pandas as pd

# =========================
# LOAD EXCEL FILE
# =========================

file_path = "your_file.xlsx"   # change this

df = pd.read_excel(file_path)

# =========================
# CLEAN DATA (recommended)
# =========================

df = df.dropna(subset=["material"])              # remove empty values
df["material"] = df["material"].astype(str).str.strip()   # remove spaces

# =========================
# COUNT TOOLS PER PART
# =========================

result = (
    df.groupby("material")
      .size()
      .reset_index(name="no_of_tools")
      .rename(columns={"material": "part"})
)

# =========================
# SORT (optional but useful)
# =========================

result = result.sort_values(by="no_of_tools", ascending=False)

# =========================
# SAVE OUTPUT
# =========================

output_path = "part_tool_count.xlsx"
result.to_excel(output_path, index=False)

print(result)

In [ ]:
import pandas as pd
import re

# =========================
# FILE PATH
# =========================
file_path = "D:/Tushar/1.Capacity Sheet ICN 01 Mar'26 update.xlsx"

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(file_path, sheet_name="ZPP016")

# =========================
# CLEAN COLUMN NAMES
# =========================
df.columns = df.columns.str.strip()

# =========================
# FILTER MOULDING AREA
# =========================
df = df[df["Area"].astype(str).str.strip().str.lower().eq("moulding")].copy()

# =========================
# CLEAN MATERIAL COLUMN
# =========================
df = df.dropna(subset=["Material"])
df["Material"] = df["Material"].astype(str).str.strip().str.upper()

# =========================
# CLEAN WORK CENTER COLUMN
# =========================
df = df.dropna(subset=["Work center name"])
df["Work center name"] = df["Work center name"].astype(str)

# =========================
# EXTRACT TONNAGE
# =========================
def extract_tonnage(text):
    text = str(text).upper()
    
    # Case 1: 120T / 120 T / 120t
    match = re.search(r'(\d+)\s*T', text)
    
    # Case 2: T120 (rare case)
    if not match:
        match = re.search(r'T\s*(\d+)', text)
    
    return int(match.group(1)) if match else None

df["Tonnage"] = df["Work center name"].apply(extract_tonnage)

# =========================
# GROUP & AGGREGATE
# =========================
result = (
    df.groupby("Material")
    .agg(
        no_of_tools=("Material", "size"),
        tonnage=("Tonnage", lambda x: ",".join(
            sorted(set(str(int(i)) for i in x if pd.notnull(i)))
        ))
    )
    .reset_index()
    .rename(columns={"Material": "part"})
)

# =========================
# SORT RESULTS
# =========================
result = result.sort_values(by="no_of_tools", ascending=False)

# =========================
# SAVE OUTPUT
# =========================
output_path = "part_tool_tonnage_moulding.xlsx"
result.to_excel(output_path, index=False)

# =========================
# DEBUG / VALIDATION
# =========================
print(result.head(20))
print("\nTotal unique parts:", result.shape[0])
print("Total tools:", result["no_of_tools"].sum())

# Optional: check extraction quality
print("\nSample extraction check:")
print(df[["Work center name", "Tonnage"]].head(20))